# Openjutsu en Google Colab

[Openjutsu](https://github.com/D3voz/openjutsu) cambia el personaje o un objeto de un vídeo con pesos abiertos (MiniMax H3, SAM 3.1 y Qwen3-VL) sobre ComfyUI. Este cuaderno lo instala en un worker de Colab, abre un túnel público y guarda la base de datos de trabajos en Google Drive para que sobreviva a los reinicios.

**Qué hace, en orden:** comprueba la GPU → prepara las claves → monta Drive → instala Node 22 → clona la rama de trabajo → restaura la base de datos → instala ComfyUI con los nodos personalizados → (opcional) descarga modelos → arranca ComfyUI, el proxy y la aplicación → abre el túnel → comprueba la API → copia la base de datos cada 5 minutos.

**Antes de empezar**

1. \`Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU\` (o mejor). Con el plan gratuito la GPU es T4 de 16 GB, que no alcanza para el flujo completo de H3.
2. Ejecuta las celdas de arriba abajo, una a una la primera vez. Las de instalación tardan varios minutos y no imprimen nada mientras trabajan.
3. **Los pesos no están incluidos.** Los modelos de MiniMax H3 (decenas de GB) no tienen descarga pública en este repositorio: hay que subirlos. La celda de modelos solo descarga los que sí tienen URL pública y está desactivada por defecto.
4. Colab no es un servidor: si cierras la pestaña o el worker queda inactivo, la sesión se detiene y el túnel deja de funcionar. Al volver a ejecutar el cuaderno, los trabajos guardados vuelven desde Drive.

Documentación de la API: [\`docs/API.es.md\`](https://github.com/D3voz/openjutsu/blob/main/docs/API.es.md).

In [ ]:
# 1. GPU, memoria y disco. Sin GPU no sigas: ComfyUI no arrancará.
import shutil
from pathlib import Path

print("=== GPU ===")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

print("\n=== PyTorch ===")
try:
    import torch
    if torch.cuda.is_available():
        print("CUDA disponible:", torch.cuda.get_device_name(0))
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("PyTorch no ve ninguna GPU.")
except Exception as exc:
    print("PyTorch no disponible:", exc)

memory = {}
for line in Path("/proc/meminfo").read_text().splitlines():
    key, _, value = line.partition(":")
    memory[key] = value.strip()
space = shutil.disk_usage("/")
print("\n=== Memoria y disco ===")
print("RAM total:", memory.get("MemTotal"), "· disponible:", memory.get("MemAvailable"))
print(f"Disco: {space.total / 1e9:.0f} GB totales, {space.free / 1e9:.0f} GB libres")

if space.free / 1e9 < 30:
    print("\nAVISO: quedan menos de 30 GB libres y los modelos de H3 necesitan decenas de GB.")

In [ ]:
# 2. Configuración y utilidades que usan las celdas siguientes.
import os
import secrets
import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlopen

REPO_URL = "https://github.com/cristhianelite2/openjutsu.git"   # tu fork
BRANCH = "espanol-colab-api"
PROJECT_DIR = Path("/content/openjutsu")
COMFY_DIR = Path("/content/ComfyUI")
LOG_DIR = Path("/content/logs")
DRIVE_DIR = Path("/content/drive/MyDrive/openjutsu")
DB_BACKUP = DRIVE_DIR / "openjutsu.db"
DB_LOCAL = PROJECT_DIR / "data" / "openjutsu.db"
COMFY_URL = "http://127.0.0.1:8188"

# Se generan de nuevo cada vez que ejecutes esta celda: cópialas antes de cerrar la sesión.
OPENJUTSU_PASSWORD = secrets.token_urlsafe(12)
OPENJUTSU_API_KEY = secrets.token_urlsafe(24)

INSTALL_MODELS = False   # True descarga solo los pesos con URL pública (no los de H3).
HF_TOKEN = ""            # opcional: token de Hugging Face para repos restringidos.

LOG_DIR.mkdir(parents=True, exist_ok=True)
PROCESSES = {}


def run(command, cwd=None, check=True):
    print("$", command, flush=True)
    result = subprocess.run(
        command, shell=True, cwd=str(cwd) if cwd else None,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    output = result.stdout.strip()
    if output:
        print(output[-3000:], flush=True)
    if check and result.returncode != 0:
        raise SystemExit(f"Falló (código {result.returncode}): {command}")
    return result


def start_background(name, command, cwd=None):
    path = LOG_DIR / f"{name}.log"
    handle = path.open("w")
    print("$", command, "+>", path, flush=True)
    process = subprocess.Popen(
        command, shell=True, cwd=str(cwd) if cwd else None,
        stdout=handle, stderr=subprocess.STDOUT, start_new_session=True,
    )
    process.log_path = path
    process.log_file = handle
    PROCESSES[name] = process
    return process


def tail(name, count=40):
    path = LOG_DIR / f"{name}.log"
    if path.exists():
        print("".join(path.read_text(errors="ignore").splitlines(keepends=True)[-count:]))


def wait_for_http(name, url, seconds=600):
    deadline = time.time() + seconds
    while time.time() < deadline:
        process = PROCESSES.get(name)
        if process is not None and process.poll() is not None:
            tail(name)
            raise SystemExit(f"{name} terminó con código {process.returncode}. Log: {process.log_path}")
        try:
            with urlopen(url, timeout=5) as response:
                status = response.status
        except HTTPError as error:
            status = error.code
        except Exception:
            status = None
        if status is not None and status < 500:
            print(f"{name} responde en {url} (HTTP {status})", flush=True)
            return
        time.sleep(3)
    tail(name)
    raise SystemExit(f"{name} no respondió en {seconds} s. Log: {LOG_DIR / (name + '.log')}")


def wait_for_port(name, host, port, seconds=300):
    deadline = time.time() + seconds
    while time.time() < deadline:
        process = PROCESSES.get(name)
        if process is not None and process.poll() is not None:
            tail(name)
            raise SystemExit(f"{name} terminó con código {process.returncode}. Log: {process.log_path}")
        try:
            with socket.create_connection((host, port), timeout=2):
                print(f"{name} escucha en {host}:{port}", flush=True)
                return
        except OSError:
            time.sleep(2)
    tail(name)
    raise SystemExit(f"{name} no abrió {host}:{port} en {seconds} s.")


print("Contraseña del navegador (usuario: openjutsu):", OPENJUTSU_PASSWORD)
print("Clave de la API (Authorization: Bearer ...):", OPENJUTSU_API_KEY)
print("Cópialas ahora: cambian si vuelves a ejecutar esta celda.")

In [ ]:
# 3. Monta Google Drive para guardar la base de datos (autoriza cuando lo pida).
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de copias:", DRIVE_DIR)
if DB_BACKUP.exists():
    print(f"Copia encontrada: {DB_BACKUP} ({DB_BACKUP.stat().st_size / 1024:.0f} KB)")
else:
    print("Todavía no hay copia en", DB_BACKUP)

In [ ]:
# 4. Node.js 22.13 o superior: la base de datos y la API usan node:sqlite.
sudo = "" if os.geteuid() == 0 else "sudo "
probe = subprocess.run("node -p process.versions.node", shell=True, text=True, capture_output=True)
version = probe.stdout.strip()
parts = [int(part) for part in version.split(".")] if version[:1].isdigit() else [0, 0, 0]
print("Node instalado:", version or "ninguno")

if len(parts) >= 3 and (parts[0] > 22 or (parts[0] == 22 and parts[1] >= 13)):
    print("No hace falta cambiarlo.")
else:
    run(f"curl -fsSL https://deb.nodesource.com/setup_22.x | {sudo}bash -")
    run(f"{sudo}apt-get install -y nodejs")
run("node --version && npm --version")

In [ ]:
# 5. Clona el fork en la rama de trabajo, o actualiza la copia que ya existe.
if (PROJECT_DIR / ".git").exists():
    run(f"git fetch origin {BRANCH}", cwd=PROJECT_DIR)
    run(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
    run(f"git reset --hard origin/{BRANCH}", cwd=PROJECT_DIR)
else:
    run(f"git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}")
run("git log --oneline -1", cwd=PROJECT_DIR)

In [ ]:
# 6. Restaura la base de datos de trabajos desde Drive, o empieza con una vacía.
import shutil

DB_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if DB_BACKUP.exists():
    shutil.copy2(DB_BACKUP, DB_LOCAL)
    print(f"Restaurada desde {DB_BACKUP}")
    for suffix in ("-wal", "-shm"):
        leftover = DB_LOCAL.with_name(DB_LOCAL.name + suffix)
        if leftover.exists():
            leftover.unlink()
else:
    print("Sin copia previa: la base de datos se creará vacía en", DB_LOCAL)
run("npm run db:info", cwd=PROJECT_DIR)

In [ ]:
# 7. ComfyUI y los nodos personalizados del README. Tarda varios minutos.
import shutil

CUSTOM_NODES = [
    ("https://github.com/ethanfel/ComfyUI-MiniMaxH3-Context-Loop", "ComfyUI-MiniMaxH3-Context-Loop"),
    ("https://github.com/Adudeguyman/ComfyUI-Fantastic-MiniMaxH3-PromptBuilder", "ComfyUI-Fantastic-MiniMaxH3-PromptBuilder"),
    ("https://github.com/kijai/ComfyUI-KJNodes", "ComfyUI-KJNodes"),
    ("https://github.com/Fannovel16/comfyui_controlnet_aux", "comfyui_controlnet_aux"),
    ("https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite", "ComfyUI-VideoHelperSuite"),
    ("https://github.com/D3voz/comfyui-h3-oneclick-swap", "comfyui-h3-oneclick-swap"),
    ("https://github.com/Carasibana/ComfyUI-H3-FaceRefine", "ComfyUI-H3-FaceRefine"),
    ("https://github.com/ltdrdata/ComfyUI-Impact-Pack", "ComfyUI-Impact-Pack"),
    ("https://github.com/D3voz/comfyui-image-detail-contrast", "comfyui-image-detail-contrast"),
    ("https://github.com/orion4d/Orion4D_FXMax", "Orion4D_FXMax"),
    ("https://github.com/D3voz/comfyui-websocket-reconnect-fix", "comfyui-websocket-reconnect-fix"),
]

if not (COMFY_DIR / "main.py").exists():
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}")
run(f"{sys.executable} -m pip install -q -r {COMFY_DIR / 'requirements.txt'}")

nodes_dir = COMFY_DIR / "custom_nodes"
nodes_dir.mkdir(parents=True, exist_ok=True)

for url, name in CUSTOM_NODES:
    target = nodes_dir / name
    if target.exists():
        run(f"git -C {target} pull --ff-only", check=False)
    else:
        print("Instalando", name, flush=True)
        run(f"git clone --depth 1 {url} {target}")
    requirements = target / "requirements.txt"
    if requirements.exists():
        result = run(f"{sys.executable} -m pip install -q -r {requirements}", check=False)
        if result.returncode != 0:
            print(f"AVISO: faltan dependencias de {name}; revisa el log de arriba.")

# SolarRenderer solo publica el nodo en su subcarpeta comfyui/.
solar_repo = Path("/content/SolarRenderer")
if not solar_repo.exists():
    run(f"git clone --depth 1 https://github.com/Enderfga/SolarRenderer {solar_repo}")
solar_target = nodes_dir / "ComfyUI-SolarRenderer"
shutil.copytree(solar_repo / "comfyui", solar_target, dirs_exist_ok=True)
solar_requirements = solar_target / "requirements.txt"
if solar_requirements.exists():
    run(f"{sys.executable} -m pip install -q -r {solar_requirements}", check=False)

print("Nodos instalados en", nodes_dir)

In [ ]:
# 8. (Opcional) Descarga los pesos que sí tienen URL pública.
# Los modelos de MiniMax H3 no tienen enlace público: ver "Modelos" en el README.
MODELS = [
    ("text_encoders/qwen3vl_8b_int8_convrot.safetensors", "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors"),
    ("text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"),
    ("text_encoders/gemma4_e2b_it_int8_convrot.safetensors", "https://huggingface.co/Comfy-Org/gemma-4/resolve/main/text_encoders/gemma4_e2b_it_int8_convrot.safetensors"),
    ("loras/solarrenderer_4step_lora.safetensors", "https://huggingface.co/Enderfga/SolarRenderer/resolve/main/solarrenderer_4step_lora.safetensors"),
    ("loras/solarrenderer_control_lora.safetensors", "https://huggingface.co/Enderfga/SolarRenderer/resolve/main/solarrenderer_control_lora.safetensors"),
    ("ultralytics/bbox/face_yolov8m.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt"),
    ("ultralytics/segm/person_yolov8m-seg.pt", "https://huggingface.co/Bingsu/adetailer/resolve/main/person_yolov8m-seg.pt"),
    ("sams/sam_vit_b_01ec64.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"),
]

MANUAL_MODELS = [
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    "loras/minimax_h3_DMAD_4step_full_lora_avg_rank_39_bf16.safetensors",
    "text_encoders/qwen3vl_32b_minimax_h3_int4_convrot.safetensors",
    "vae/minimax_h3_video_vae_int8_convrot.safetensors",
    "vae/minimax_h3_audio_vae_fp32.safetensors",
    "checkpoints/sam3.1_multiplex_fp16.safetensors",
]

if not INSTALL_MODELS:
    print("Paso omitido: pon INSTALL_MODELS = True en la celda 2 y vuelve a ejecutarla.")
    print("Estos archivos de MiniMax H3 no tienen URL pública y hay que subirlos a mano:")
    for name in MANUAL_MODELS:
        print("  ComfyUI/models/" + name)
else:
    import urllib.request

    headers = {"User-Agent": "openjutsu-colab"}
    if HF_TOKEN:
        headers["Authorization"] = "Bearer " + HF_TOKEN
    for relative, url in MODELS:
        target = COMFY_DIR / "models" / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            print("Ya está:", relative, flush=True)
            continue
        print("Descargando", relative, flush=True)
        request = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(request) as response, target.open("wb") as handle:
            shutil.copyfileobj(response, handle)
        print(f"  {target.stat().st_size / 1e9:.2f} GB", flush=True)
    print("\nRecuerda: faltan los pesos de H3, que no tienen URL pública:")
    for name in MANUAL_MODELS:
        print("  ComfyUI/models/" + name)

print("Espacio libre:", f"{shutil.disk_usage('/').free / 1e9:.0f} GB")

In [ ]:
# 9. Arranca ComfyUI en 127.0.0.1:8188 y espera a que responda.
existing = PROCESSES.get("comfyui")
if existing is not None and existing.poll() is None:
    print("ComfyUI ya está arrancado en esta sesión.")
else:
    start_background("comfyui", f"{sys.executable} main.py --listen 127.0.0.1 --port 8188", cwd=COMFY_DIR)
wait_for_http("comfyui", COMFY_URL + "/system_stats", seconds=900)
tail("comfyui", 20)

In [ ]:
# 10. Instala comfy-api-proxy 0.1.6, aplica el parche de progreso en vivo y lo arranca en 8189.
run(f"{sys.executable} -m pip install -q comfy-api-proxy==0.1.6")
run(f"{sys.executable} patch_proxy.py", cwd=PROJECT_DIR)
existing = PROCESSES.get("proxy")
if existing is not None and existing.poll() is None:
    print("El proxy ya está arrancado.")
else:
    start_background("proxy", "comfy-api-proxy run --max-upload-mb 500")
wait_for_port("proxy", "127.0.0.1", 8189)
tail("proxy", 20)

In [ ]:
# 11. Escribe .env.local, compila y arranca la aplicación en 127.0.0.1:3000.
env_file = PROJECT_DIR / ".env.local"
env_file.write_text("\n".join([
    "COMFY_BASE_URL=http://127.0.0.1:8189",
    "COMFY_API_KEY=",
    "COMFYUI_URL=http://127.0.0.1:8188",
    f"OPENJUTSU_PASSWORD={OPENJUTSU_PASSWORD}",
    f"OPENJUTSU_API_KEY={OPENJUTSU_API_KEY}",
    f"OPENJUTSU_DB_PATH={DB_LOCAL}",
    "",
]))
print(env_file.read_text().replace(OPENJUTSU_PASSWORD, "********").replace(OPENJUTSU_API_KEY, "********"))

run("npm ci --no-audit --no-fund", cwd=PROJECT_DIR)
run("npm run build", cwd=PROJECT_DIR)

existing = PROCESSES.get("app")
if existing is not None and existing.poll() is None:
    existing.terminate()
    existing.wait(timeout=30)
start_background("app", "npm start", cwd=PROJECT_DIR)
wait_for_http("app", "http://127.0.0.1:3000/api/v1/health", seconds=180)
run("npm run db:info", cwd=PROJECT_DIR)

In [ ]:
# 12. Abre un túnel público con cloudflared y muestra la URL.
import re

CLOUDFLARED = Path("/usr/local/bin/cloudflared")
if not CLOUDFLARED.exists():
    run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    run(f"chmod +x {CLOUDFLARED}")

existing = PROCESSES.get("tunnel")
if existing is not None and existing.poll() is None:
    print("El túnel ya está abierto.")
else:
    start_background("tunnel", f"{CLOUDFLARED} tunnel --no-autoupdate --url http://127.0.0.1:3000")

PUBLIC_URL = None
for _ in range(90):
    log = LOG_DIR / "tunnel.log"
    text = log.read_text(errors="ignore") if log.exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(1)

if PUBLIC_URL:
    print("\nOpenjutsu está en:", PUBLIC_URL)
    print("Usuario: openjutsu")
    print("Contraseña:", OPENJUTSU_PASSWORD)
    print("Clave de la API:", OPENJUTSU_API_KEY)
else:
    tail("tunnel", 30)
    print("No se encontró la URL del túnel todavía.")

In [ ]:
# 13. Comprueba la API v1 con la clave y muestra el estado guardado.
import json
from urllib.error import HTTPError
from urllib.request import Request, urlopen


def api(path, method="GET"):
    request = Request(
        "http://127.0.0.1:3000" + path,
        method=method,
        headers={"Authorization": "Bearer " + OPENJUTSU_API_KEY},
    )
    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


try:
    print(json.dumps(api("/api/v1/health"), indent=2, ensure_ascii=False))
    print(json.dumps(api("/api/v1/stats"), indent=2, ensure_ascii=False))
    recent = api("/api/v1/jobs?limit=5")
    print(f"Trabajos guardados: {recent['total']}")
    for job in recent["jobs"]:
        print(f"  {job['createdAt']}  {job['status']:<10} {job['kind']:<12} {job['id']}")
except HTTPError as error:
    print("La API respondió", error.code, error.read().decode(errors="ignore"))

In [ ]:
# 14. Copia la base de datos a Drive ahora y cada 5 minutos mientras viva la sesión.
import threading

BACKUP_SECONDS = 300


def save_database(quiet=False):
    result = subprocess.run(
        ["npm", "run", "db:backup", "--", str(DB_BACKUP)],
        cwd=str(PROJECT_DIR), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if not quiet:
        print(result.stdout.strip())
    return result.returncode == 0


def backup_loop():
    while True:
        time.sleep(BACKUP_SECONDS)
        try:
            ok = save_database(quiet=True)
            print(("Copia guardada" if ok else "Copia fallida") + " en " + str(DB_BACKUP), flush=True)
        except Exception as exc:
            print("Error al copiar la base de datos:", exc, flush=True)


save_database()
threading.Thread(target=backup_loop, daemon=True, name="db-backup").start()
print(f"Copias automáticas cada {BACKUP_SECONDS} s en {DB_BACKUP}")
print("Guarda una copia a mano cuando quieras con save_database().")

## Usar la API

Con la URL del túnel y la clave de la celda 2:

~~~sh
# Estado
curl -H "Authorization: Bearer $OPENJUTSU_API_KEY" "$PUBLIC_URL/api/v1/health"

# Enviar un trabajo (multipart, como la interfaz)
curl -H "Authorization: Bearer $OPENJUTSU_API_KEY" \
  -F "mode=simple" -F "video=@video.mp4" -F "photos=@persona.jpg" \
  "$PUBLIC_URL/api/v1/jobs"

# Consultar un trabajo
curl -H "Authorization: Bearer $OPENJUTSU_API_KEY" "$PUBLIC_URL/api/v1/jobs/<id>"
~~~

Los campos de cada modo y los códigos de error están en [\`docs/API.es.md\`](https://github.com/D3voz/openjutsu/blob/main/docs/API.es.md), con ejemplos en Python.

## Volver a arrancar tras un reinicio

Un reinicio del worker borra \`/content\`: hay que repetir las celdas **2 a 14** (la 8 solo si quieres modelos). La base de datos vuelve desde Drive en la celda 6, así que la lista de trabajos y sus resultados siguen ahí.

Si no quieres volver a descargar los modelos en cada sesión, guárdalos en Drive y enlaza la carpeta antes de la celda 9:

~~~python
import shutil
shutil.rmtree(COMFY_DIR / "models", ignore_errors=True)
(COMFY_DIR / "models").symlink_to(DRIVE_DIR / "models", target_is_directory=True)
~~~

Sigue siendo más lento que el disco del worker: úsalo para ahorrar descargas, no para trabajar a diario.

## Límites de Colab que conviene saber

- La sesión se corta a las ~12 h y tras ~90 min de inactividad. El túnel y la aplicación mueren con ella.
- El plan gratuito da una T4 de 16 GB y unos 80 GB de disco. El flujo completo de H3 no cabe: usa el cuaderno para probar la interfaz y la API, y una GPU local o de pago para renderizar de verdad.
- El túnel de \`trycloudflare\` es público: cualquiera con la URL llega al login. La contraseña protege el acceso, pero no compartas la URL ni la clave de la API.
- Las claves se generan en cada sesión. Si quieres una fija, escribe la tuya en la celda 2 (mínimo 12 caracteres para la contraseña).